# The festival fund fiasco
How much cash is available, can the DJ be paid, and what explains the difference?

Read `docs/student-brief.md` first. Work as of **11 June 2026 at 12:00**; opening balance is **0 DKK**. Keep raw evidence unchanged.

**About 30 minutes:** inspect (4 min), clean (7 min), reconcile (8 min), follow evidence (7 min), verdict (4 min). These are suggested time boxes, not a deadline.

File loading, duplicate handling, and evidence display are provided. Your coding tasks are **eight marked lines and one short loop body**, using functions, dictionaries, loops, and `if` statements. Replace `...` with Python code, and replace the marked `pass` with your own lines. Finish each section before running it with **Shift + Enter**. There are also **two deliberate bugs** to find and fix in addition to the existing tasks. The affected sections have a warning, but the faulty lines are not marked. Complete the tasks, run the cells, and check the code and output for mistakes. No extra packages are needed.

Read the output and discuss your findings aloud. Completing the full evidence table and writing a 3–5 sentence verdict are optional; you can discuss both aloud instead during the 30-minute exercise. Charts and joins are optional extensions outside the 30 minutes.

**Team:** ...  **Our initial hypothesis (not a conclusion):** ...


In [ ]:
import csv
from decimal import Decimal
from pathlib import Path

# Setup is provided. Run this cell without changing it.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "data"

def load_csv(filename):
    with (DATA / filename).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

payments_raw = load_csv("payments.csv")
receipts = load_csv("receipts.csv")
chat = load_csv("chat.csv")
print("Payment rows:", len(payments_raw))
print("Receipt rows:", len(receipts), "| Chat messages:", len(chat))
payments_raw[:3]


## 1. Inspect before cleaning · 4 minutes
**Debugging warning:** this section contains one deliberate bug. Find and fix it.

Run the next cell to inspect the columns, statuses, empty fields, and repeated payment IDs without changing the original dictionaries.

Discuss: which differences affect cash? Which are harmless? A blank optional message is valid.


In [ ]:
print("Columns:", list(payments_raw[0]))

for row in payments_raw:
    statuses = []
    if row["status"] not in statuses:
        statuses.append(row["status"])
print("Statuses:", statuses)

print("\nEmpty fields:")
for row in payments_raw:
    for column in row:
        if row[column].strip() == "":
            print(row["payment_id"], column)

print("\nRepeated payment IDs:")
seen_rows = {}
for row in payments_raw:
    payment_id = row["payment_id"]
    if payment_id in seen_rows:
        print(payment_id, "| identical:", row == seen_rows[payment_id])
    else:
        seen_rows[payment_id] = row


## 2. Clean and deduplicate · 7 minutes
Complete **tasks 1–3** in the next cell:

1. Write the return expression in `parse_money`. Remove surrounding spaces, replace commas with decimal points, and convert to `Decimal`. Useful methods: `.strip()` and `.replace(old, new)`.
2. Make the cleaned status lowercase using `.lower()`.
3. Use your `parse_money` function to convert the cleaned amount.

The surrounding loop is provided. It copies each field into a new dictionary, leaving the original evidence unchanged. `Decimal` stores money without rounding surprises.

In section 1, both repeated IDs should show `identical: True`. If an ID shows `False`, stop here and discuss the conflicting records with your instructor. The code below skips only exact copies of whole rows.

After running, explain why removing rows just because they share a sender or reference would lose real information.


In [ ]:
def parse_money(text):
    return ...  # Task 1: strip spaces, replace the comma, convert to Decimal.

payments_clean = []
unique_raw = []
removed_ids = []

for row in payments_raw:
    if row in unique_raw:
        removed_ids.append(row["payment_id"])
    else:
        unique_raw.append(row)
        cleaned = {}
        for column in row:
            cleaned[column] = row[column].strip()
        cleaned["status"] = ...  # Task 2
        cleaned["amount_dkk"] = ...  # Task 3
        payments_clean.append(cleaned)

print("Raw rows:", len(payments_raw))
print("Cleaned rows:", len(payments_clean))
print("Removed exact duplicate copies:", removed_ids)
print("Example cleaned payment:", payments_clean[0])


**Cleaning log:** We normalized ... and removed ... because ... . We kept the blank message because ... . Two payments from the same sender can be valid because ... .

## 3. Reconcile current cash · 8 minutes
Complete **tasks 4–6** and the short loop body below.

- Task 4: calculate `balance` from `opening_balance`, `income`, and `spending`.
- Task 5: find the DJ invoice by checking its `reference` and `status`.
- Task 6: calculate `difference` as cash available minus the DJ amount.

Inside the payment loop, replace `pass` with two `if` statements and their indented additions. One should add incoming money to `income`; the other should add outgoing money to `spending`. Work out which of `sender` and `recipient` to compare with `FUND`. Use `payment["amount_dkk"]` for the amount.

Only completed payments count. Amounts are positive magnitudes. Receipts describe purchases or invoices; they are not extra payments.

Before running, predict: would including a pending refund overstate or understate available cash?


In [ ]:
FUND = "Festival Fund"
opening_balance = Decimal("0.00")
income = Decimal("0.00")
spending = Decimal("0.00")

for payment in payments_clean:
    if payment["status"] == "completed":
        # Replace pass with your two if statements and their additions.
        pass

balance = ...  # Task 4

for receipt in receipts:
    if ...:  # Task 5: reference DJ-01 AND status unpaid.
        dj_amount = parse_money(receipt["amount_dkk"])
        dj_deadline = receipt["expected_date"]

difference = ...  # Task 6

print(f"Completed income: {income:.2f} DKK")
print(f"Completed spending: {spending:.2f} DKK")
print(f"Cash available: {balance:.2f} DKK")
print(f"Unpaid DJ invoice: {dj_amount:.2f} DKK")
print("DJ deadline:", dj_deadline)
if difference < 0:
    print(f"Shortfall: {-difference:.2f} DKK")
else:
    print(f"Surplus after paying DJ: {difference:.2f} DKK")

print("\nPending payments — excluded from current cash:")
for payment in payments_clean:
    if payment["status"] == "pending":
        print(payment["reference"], payment["sender"], "→", payment["recipient"],
              f'{payment["amount_dkk"]:.2f} DKK')


## 4. Follow references and read the chat · 7 minutes
**Run the next cell as provided.** It groups payments and receipts for the three references below, then displays the chat. A shared reference connects evidence; it does not make a purchase and refund the same transaction.

Use the output to discuss the evidence aloud. **Optional written task:** fill the table in your own words if time allows.

| Reference | Cash direction and status | Receipt/chat evidence | Meaning |
| --- | --- | --- | --- |
| FLAMINGOS-01 | ... | ... | ... |
| SPEAKER-01 | ... | ... | ... |
| DJ-01 | ... | ... | ... |

Who reimbursed Oliver? Who still holds the speaker deposit? Compare the refund's expected settlement with the DJ deadline. A receipt's `expected_date` is a purchase date for a paid receipt or a deadline for an unpaid invoice, not a refund settlement date.


In [ ]:
for reference in ["FLAMINGOS-01", "SPEAKER-01", "DJ-01"]:
    print("\nREFERENCE:", reference)
    payment_found = False

    for payment in payments_clean:
        if payment["reference"] == reference:
            payment_found = True
            print("Payment:", payment["sender"], "→", payment["recipient"],
                  f'{payment["amount_dkk"]:.2f} DKK', payment["status"])

    if payment_found == False:
        print("No payment recorded.")

    for receipt in receipts:
        if receipt["reference"] == reference:
            print("Receipt:", receipt["receipt_id"])
            print("Buyer:", receipt["buyer"], "| Vendor:", receipt["vendor"])
            print("Amount:", receipt["amount_dkk"], "DKK | Status:", receipt["status"])
            print("Purchase date / invoice deadline:", receipt["expected_date"])
            print("Description:", receipt["description"])

print("\nCOMMITTEE CHAT:")
for message in chat:
    print(message["timestamp"], message["sender"] + ":", message["message"])


## 5. Conditional scenario and verdict · 4 minutes

**Debugging warning:** this section contains one deliberate bug in addition to the tasks below. Find and fix it.
Complete **tasks 7–8** in the next cell:

- Task 7: choose the refund's status in the condition. Use the cleaned payment data to decide.
- Task 8: calculate what would remain after paying the DJ if the refund settles.

Compare the settlement timing in the chat with the DJ deadline. Does having enough money later mean the DJ can be paid on time?

Discuss the following aloud. **Optional written task:** write a 3–5 sentence verdict if time allows.

- Current cash, DJ invoice, and surplus or shortfall.
- The conditional refund scenario and its timing.
- What the speaker payment actually represents.
- A practical next step and one uncertainty.
- Whether these records establish theft.

Use amounts and references from your output. Separate evidence from assumptions.

**Our verdict (optional to write; you can discuss it aloud):** ...


In [ ]:
refund_amount = Decimal("0.00")

for payment in payments_clean:
    if payment["reference"] == "FLAMINGOS-01":
        if payment["recipient"] == FUND and payment["status"] == ...:  # Task 7
            refund_amount = refund_amount + payment["amount_dkk"]

conditional_balance = balance - refund_amount
conditional_remaining = ...  # Task 8

print("CONDITIONAL: IF the approved flamingo refund settles")
print(f"Current cash: {balance:.2f} DKK")
print(f"Refund: {refund_amount:.2f} DKK")
print(f"Cash after settlement: {conditional_balance:.2f} DKK")
print(f"Remaining after paying DJ: {conditional_remaining:.2f} DKK")
print("This scenario is not the current balance.")
print("Check the chat: would settlement happen before the DJ deadline?")


## Optional: visualize or merge · extra time
The core investigation is complete once you can explain your verdict aloud or in writing. If you have extra time, make a labelled bar chart of completed spending by reference with Matplotlib, or use Pandas to join outgoing payments to receipts. These extensions require extra packages. Can you validate the join as one-to-one? What happens if you include the pending refund in the same join?

Restart the kernel and run your completed cells in order to check your work. You are done when you can explain your findings. There is nothing to hand in. For help, open `docs/hints.md`.


In [ ]:
# Optional stretch work goes here.
